# Beacon qualification SLM on Kaggle (T4) — raw data → QLoRA adapter → evaluation table

Settings: **Accelerator = GPU T4**, **Internet = On**. Input: the `beacon.zip` dataset.
Use **Save Version → Save & Run All** so it runs on Kaggle's server even with the browser closed.
Outputs land in `/kaggle/working/` (Output tab): `beacon_run.zip`.

In [ ]:
import os, sys, glob, json, shutil, subprocess, time, zipfile
NOTEBOOK_STARTED = time.time()
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one GPU: QLoRA with device_map={'': 0}
work = '/kaggle/working/beacon'
if not os.path.exists(work):
    found = glob.glob('/kaggle/input/**/slm/train.py', recursive=True)
    if found: shutil.copytree(os.path.dirname(os.path.dirname(found[0])), work)
    else: zipfile.ZipFile(glob.glob('/kaggle/input/**/*.zip', recursive=True)[0]).extractall(work)
os.chdir(work); sys.path.insert(0, work)
def sh(args):
    # run a command and stream its output into the cell so progress stays visible
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in proc.stdout: print(line, end='', flush=True)
    if proc.wait(): raise RuntimeError(f'failed ({proc.returncode}): ' + ' '.join(args))
sh([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'pydantic'])
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
for script in sorted(glob.glob('slm/teacher/gen*.py'), key=lambda p: int(''.join(filter(str.isdigit, os.path.basename(p))))):
    subprocess.run([sys.executable, os.path.basename(script)], cwd='slm/teacher', check=True)
print(subprocess.run([sys.executable, 'slm/build.py'], capture_output=True, text=True).stdout[:1500])

In [ ]:
sh([sys.executable, 'slm/train.py', '--out', 'slm/runs/kaggle', '--epochs', '2'])
shutil.make_archive('/kaggle/working/adapter_backup', 'zip', '.', 'slm/runs/kaggle/adapter')

In [ ]:
EVAL = 'slm/eval/eval_set.jsonl' if os.path.exists('slm/eval/eval_set.jsonl') else 'slm/data/dev.jsonl'
print('evaluating on', EVAL, '(dev.jsonl = teacher labels: development numbers only)' if 'dev' in EVAL else '')
for system, extra in [('B', []), ('C', ['--adapter', 'slm/runs/kaggle/adapter'])]:
    sh([sys.executable, 'slm/evaluate.py', 'run', '--system', system, '--backend', 'hf', '--eval', EVAL, '--gpu-hourly', '0'] + extra)
sh([sys.executable, 'slm/evaluate.py', 'table'])

In [ ]:
meta = json.load(open('slm/runs/kaggle/run_metadata.json'))
meta['notebook_seconds'] = round(time.time() - NOTEBOOK_STARTED); meta['eval_file'] = EVAL
json.dump(meta, open('slm/runs/kaggle/run_metadata.json', 'w'), indent=1)
with zipfile.ZipFile('/kaggle/working/beacon_run.zip', 'w') as z:
    for f in glob.glob('slm/runs/kaggle/adapter/*') + ['slm/runs/kaggle/run_metadata.json'] + glob.glob('slm/results/*'): z.write(f)
shutil.rmtree('/kaggle/working/beacon')   # keep the Output tab to the two zips
print('done: download beacon_run.zip from the Output tab')